# Notebook 01 — Inspección Inicial
## Proyecto Integrador | Minería de Datos I
**Integrante:** Val Martinetti  
**Dataset:** `streaming_users_dirty.json` — Usuarios de plataforma de streaming  

---
### Objetivo
Reunir evidencia sobre la estructura y el estado del dataset **sin tomar decisiones de limpieza todavía**. El objetivo es responder: ¿qué tenemos?, ¿qué problemas se observan?, ¿qué preguntas se abren?

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar dataset crudo (JSON)
with open('../data/raw/streaming_users_dirty.json') as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")

## 1. Estructura general

In [ ]:
# Primeras filas
df.head(10)

In [ ]:
# Tipos de dato y valores no nulos
df.info()

In [ ]:
# Clasificación de variables
print("="*55)
print("VARIABLES CUANTITATIVAS (numéricas):"  )
print("  - user_id         : Discreta (identificador)")
print("  - age             : Discreta (años)")
print("  - monthly_watch_time_mins: Continua (minutos/mes)")
print("  - customer_support_tickets: Discreta (conteo)")
print()
print("VARIABLES CUALITATIVAS (categóricas):")
print("  - subscription_plan: Nominal (Básico/Estándar/Premium)")
print("  - country           : Nominal (7 países)")
print("  - favorite_genre    : Nominal (7 géneros)")
print("  - last_login_date   : Fecha (string → datetime)")

## 2. Estadísticos descriptivos

In [ ]:
# Resumen numérico completo
df.describe(include='all').round(2)

**Observaciones clave de `describe()`:**
- `age`: min = **-5**, max = **150** → valores claramente imposibles. Rango esperado para una plataforma de streaming: 6–100 años.
- `monthly_watch_time_mins`: min = **-120** (imposible) y max = **99 999** (extremo sospechoso). Hay 193 nulos.
- `customer_support_tickets`: min = **-1** (imposible), max = **150** (outlier extremo dado que el 75% tiene ≤ 1).
- `last_login_date`: fecha más frecuente `2026-15-03` → **mes 15 no existe**; hay fechas inválidas embebidas.
- `subscription_plan` / `country` / `favorite_genre`: `unique` cuenta variantes que representan la misma categoría (p.ej. "Básico", "basico", "BASICO", "Basic").

## 3. Datos faltantes

In [ ]:
# Conteo y porcentaje de faltantes por columna
faltantes = df.isnull().sum()
porcentaje = (faltantes / len(df) * 100).round(2)
resumen = pd.DataFrame({'Faltantes': faltantes, '% del total': porcentaje})
print(resumen.sort_values('Faltantes', ascending=False))

print(f"\nTotal de celdas nulas: {faltantes.sum()} de {df.shape[0]*df.shape[1]} ({faltantes.sum()/(df.shape[0]*df.shape[1])*100:.2f}%)")

**Observaciones:**
- `last_login_date`: **3.92%** de nulos. Adicionalmente, algunas fechas presentes son inválidas (mes > 12) → el % real de "no utilizable" es mayor.
- `favorite_genre`: **2.94%** de nulos → necesita diagnóstico de mecanismo.
- `monthly_watch_time_mins`: **2.37%** nulos → se investigará si depende del plan.
- `age`, `customer_support_tickets`, `subscription_plan`, `country`: **0 nulos explícitos**, pero tienen valores imposibles que deberán convertirse a NaN en la limpieza.

## 4. Duplicados

In [ ]:
# Registros completamente duplicados (excepto user_id)
n_dup = df.drop_duplicates(subset=df.columns.difference(['user_id'])).shape[0]
duplicados = len(df) - n_dup
print(f"Registros duplicados (ignorando user_id): {duplicados} ({duplicados/len(df)*100:.2f}%)")
print()
# Ver algunos duplicados
dup_mask = df.duplicated(subset=df.columns.difference(['user_id']), keep=False)
print("Ejemplo de filas duplicadas:")
print(df[dup_mask].sort_values('age').head(6).to_string())

## 5. Problemas en variables categóricas

In [ ]:
# Inspección de variantes en cada categórica
for col in ['subscription_plan', 'country', 'favorite_genre']:
    print(f"\n{'='*50}")
    print(f"  {col} — {df[col].nunique(dropna=False)} variantes únicas:")
    print(df[col].value_counts(dropna=False).to_string())

**Resumen de inconsistencias categóricas:**
- `subscription_plan`: 15 variantes para 3 valores reales (Básico/Estándar/Premium). Problemas: mayúsculas, tildes, abreviaturas ("Std", "STANDARD"), errores tipográficos ("Premiun").
- `country`: 26 variantes para 7 países. Problemas: mayúsculas, idioma (Brazil/Brasil), abreviaturas ISO (ARG, BRA, MEX…), espacios finales.
- `favorite_genre`: 28 variantes para 7 géneros, más 240 nulos. Problemas: mayúsculas, idioma (Action/Acción, Comedy/Comedia), errores tipográficos ("thriler"), abreviaturas ("DOC"), equivalencias entre idiomas ("Crimen"/"Crime").

## 6. Problemas en variables numéricas

In [ ]:
# Detección de valores imposibles
print("AGE:")
print(f"  Negativos (< 0): {(df['age'] < 0).sum()}")
print(f"  Mayores de 100: {(df['age'] > 100).sum()}")
print(f"  Rango observado: [{df['age'].min()}, {df['age'].max()}]")

print("\nMONTHLY_WATCH_TIME_MINS:")
print(f"  Negativos (< 0): {(df['monthly_watch_time_mins'] < 0).sum()}")
print(f"  Nulos: {df['monthly_watch_time_mins'].isnull().sum()}")
print(f"  Máximo: {df['monthly_watch_time_mins'].max():.1f} (99 999 = valor centinela sospechoso)")

print("\nCUSTOMER_SUPPORT_TICKETS:")
print(f"  Negativos (< 0): {(df['customer_support_tickets'] < 0).sum()}")
print(f"  Distribución:")
print(df['customer_support_tickets'].value_counts().sort_index().to_string())

## 7. Problema en variable fecha

In [ ]:
# Detectar fechas inválidas al parsear
fechas_parseadas = pd.to_datetime(df['last_login_date'], errors='coerce')
inválidas = df['last_login_date'].notna() & fechas_parseadas.isna()
print(f"Fechas inválidas (no parseables): {inválidas.sum()}")
print(f"Total no utilizables (nulos + inválidas): {fechas_parseadas.isna().sum()}")
print()
print("Ejemplos de fechas inválidas:")
print(df.loc[inválidas, 'last_login_date'].value_counts().head(10).to_string())

## 8. Preguntas abiertas para el análisis

In [ ]:
print("""
PREGUNTAS QUE SURGEN DE LA INSPECCIÓN:
=======================================
1. ¿Cómo se distribuye el tiempo de visualización mensual según el plan de suscripción?
   → Hipótesis: usuarios Premium ven más contenido.

2. ¿Existe relación entre edad y minutos de visualización?
   → Hipótesis: jóvenes consumen más.

3. ¿Cuál es la distribución geográfica de usuarios y planes?
   → ¿Hay concentración en algún país?

4. ¿Qué géneros dominan y varían según el plan?
   → ¿Premium prefiere géneros de nicho como Documental?

5. ¿Los tickets de soporte están asociados a algún plan o país?
   → Usuarios con problemas técnicos frecuentes, ¿están en algún segmento?

PROBLEMAS A RESOLVER EN LIMPIEZA (notebook 02):
================================================
1. Normalizar categóricas: plan, país, género (15/26/28 variantes → 3/7/7 categorías)
2. Eliminar duplicados exactos (sin user_id): ~126 filas
3. age: valores < 6 y > 100 → NaN → imputar (mecanismo MCAR: falta uniforme entre planes)
4. watch_time: negativos → NaN; valor 99.999 → extremo, winsorizar k=3
5. tickets: negativos → NaN; 99 y 150 → outliers extremos, winsorizar k=3
6. last_login_date: parsear → fechas inválidas quedan NaT (no imputables)
7. favorite_genre: 240 nulos + valores None → diagnóstico + imputar con moda
""")

## Resumen de la inspección

| Problema detectado | Cantidad | Acción planeada |
|---|---|---|
| Duplicados exactos | 126 | Eliminar |
| Variantes en `subscription_plan` | 15 → 3 | Normalizar |
| Variantes en `country` | 26 → 7 | Normalizar |
| Variantes en `favorite_genre` | 28 → 7 | Normalizar |
| `age` imposibles (< 6 o > 100) | 120 | NaN → imputar (mediana) |
| `monthly_watch_time_mins` negativos | 49 | NaN → imputar |
| `monthly_watch_time_mins` nulos originales | 193 | Imputar por plan |
| `monthly_watch_time_mins` extremo (99 999) | múltiples | Winsorizar k=3 |
| `customer_support_tickets` negativos | 29 | NaN → imputar |
| `customer_support_tickets` extremos (99, 150) | 81 | Winsorizar k=3 |
| `last_login_date` nulos | 320 | Dejar NaT |
| `last_login_date` inválidas | 449 | Dejar NaT |
| `favorite_genre` nulos | 240 | Imputar con moda |

> **Nota metodológica:** en este notebook solo se documentaron los problemas. Ninguna decisión de transformación se aplicó al dataset. La limpieza completa se realiza en el `02_calidad_y_limpieza.ipynb`.